In [1]:
# === SESSION BOOTSTRAP (run first in every notebook) ======================
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys

PARENT = "/content/drive/MyDrive/UAV_TRUST_Research"
REPO   = f"{PARENT}/uav-trust-research"

for fname in (".gitconfig", ".git-credentials"):
    src = os.path.join(PARENT, fname)
    if os.path.exists(src):
        subprocess.run(f'cp "{src}" /root/{fname}', shell=True)
subprocess.run("git config --global credential.helper store", shell=True)

r = subprocess.run("git config --global user.name && git config --global user.email",
                   shell=True, capture_output=True, text=True)
print("git identity:", r.stdout.strip() or "MISSING - run 00_setup.ipynb first")

if os.path.isdir(REPO):
    os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    print("cwd:", os.getcwd())
else:
    print("Repository not on Drive yet - run 00_setup.ipynb first.")

Mounted at /content/drive
git identity: Md Anas Biswas
anasbiswas@gmail.com
cwd: /content/drive/MyDrive/UAV_TRUST_Research/uav-trust-research


In [2]:
!pip install xgboost scikit-learn matplotlib pandas numpy scipy requests --quiet

In [3]:
# Configuration for UAV-NIDD (Case 1 file; confirmed label, benign class, and leakage drops)
CONFIG = {
    "zenodo_record": "15125851",     # UAV-NIDD (open, CC-BY-4.0)
    "data_dir": "data/uav_nidd",
    "data_file": "data/uav_nidd/UAV-NDD CSV/UAV-Case1-Label.csv",  # the one clean CSV (latin-1)
    "encoding": "latin-1",
    "label_col": "Label",
    "normal_value": "Normal",
    "include_families": ["DDoS", "UDP Flooding", "MITM", "Jamming", "BruteForce", "De-authentication"],
    "subsample_n": None,           # stratified cap for a tractable first run (None = use all)
    "drop_col_patterns": [
        "unnamed", "index",
        "ip.src", "ip.dst", "srcport", "dstport", "udp.srcport", "udp.dstport",
        "frame.time", "frame.number", "time_epoch", "time_relative", "time_delta",
        "bssid", "mactime", "vendor_oui", "wlan_radio.timestamp", "wlan_radio.start_tsf",
        "radiotap.timestamp", "wlan.seq",
    ],
    "seeds": list(range(10)),         # 5 seeds for the first pass (raise to 10 to match UAVIDS)
    "conformal_alpha": 0.10,
    "n_ece_bins": 15,
    "normal_fracs": {"train": 0.60, "cal": 0.20, "test_seen": 0.10, "test_shift": 0.10},
    "family_fracs": {"train": 0.60, "cal": 0.20, "test_seen": 0.20},
    "xgb": {"n_estimators": 300, "max_depth": 6, "learning_rate": 0.1,
            "subsample": 0.9, "colsample_bytree": 0.9, "tree_method": "hist"},
    "fig_dir": "figures",
    "report_dir": "reports",
}
for d in [CONFIG["data_dir"], CONFIG["fig_dir"], CONFIG["report_dir"]]:
    os.makedirs(d, exist_ok=True)
CONFIG

{'zenodo_record': '15125851',
 'data_dir': 'data/uav_nidd',
 'data_file': 'data/uav_nidd/UAV-NDD CSV/UAV-Case1-Label.csv',
 'encoding': 'latin-1',
 'label_col': 'Label',
 'normal_value': 'Normal',
 'include_families': ['DDoS',
  'UDP Flooding',
  'MITM',
  'Jamming',
  'BruteForce',
  'De-authentication'],
 'subsample_n': None,
 'drop_col_patterns': ['unnamed',
  'index',
  'ip.src',
  'ip.dst',
  'srcport',
  'dstport',
  'udp.srcport',
  'udp.dstport',
  'frame.time',
  'frame.number',
  'time_epoch',
  'time_relative',
  'time_delta',
  'bssid',
  'mactime',
  'vendor_oui',
  'wlan_radio.timestamp',
  'wlan_radio.start_tsf',
  'radiotap.timestamp',
  'wlan.seq'],
 'seeds': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
 'conformal_alpha': 0.1,
 'n_ece_bins': 15,
 'normal_fracs': {'train': 0.6,
  'cal': 0.2,
  'test_seen': 0.1,
  'test_shift': 0.1},
 'family_fracs': {'train': 0.6, 'cal': 0.2, 'test_seen': 0.2},
 'xgb': {'n_estimators': 300,
  'max_depth': 6,
  'learning_rate': 0.1,
  'subsample': 0

In [4]:
# Imports (shared logic in src/)
import numpy as np, pandas as pd, requests, glob
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import (accuracy_score, f1_score,
                             balanced_accuracy_score, recall_score)
from src.trust import (top_label_ece, brier_binary, conformal_qhat, coverage_size,
                       aurc, logit, fit_calibrators, apply_calibrators)
from src.data import load_csvs, detect_schema, prepare_splits
print("imports ready")

imports ready


In [5]:
# Download UAV-NIDD from Zenodo (single zipped CSV bundle, ~302 MB) and unzip
rec = CONFIG["zenodo_record"]
if not glob.glob(os.path.join(CONFIG["data_dir"], "**/*.csv"), recursive=True):
    meta = requests.get(f"https://zenodo.org/api/records/{rec}", timeout=60).json()
    for f in meta.get("files", []):
        name, url = f["key"], f["links"]["self"]
        if name.lower().endswith((".csv", ".zip", ".gz")):
            dest = os.path.join(CONFIG["data_dir"], name)
            print("downloading", name, "...")
            open(dest, "wb").write(requests.get(url, timeout=1200).content)
    for z in glob.glob(os.path.join(CONFIG["data_dir"], "*.zip")):
        import zipfile; zipfile.ZipFile(z).extractall(CONFIG["data_dir"]); print("unzipped", os.path.basename(z))
print("csv files:", [os.path.basename(x) for x in glob.glob(os.path.join(CONFIG['data_dir'], '**/*.csv'), recursive=True)][:20])

csv files: ['GSC Case3 Label .csv', 'UAV-Case1-Label.csv']


In [6]:
# Load the single clean Case-1 CSV directly (latin-1). The other file in the bundle is an
# .xlsx mislabelled as .csv, so we do not use the folder-wide loader here.
import pandas as pd
assert os.path.exists(CONFIG["data_file"]), "Run the download cell first; expected %s" % CONFIG["data_file"]
df = pd.read_csv(CONFIG["data_file"], low_memory=False, encoding=CONFIG["encoding"])
label_col, normal_value = CONFIG["label_col"], CONFIG["normal_value"]
families = [v for v in df[label_col].unique() if v != normal_value]
print("full shape:", df.shape)
print("label column:", label_col, "| normal value:", repr(normal_value))
print("class distribution:")
for k, v in df[label_col].value_counts().items():
    print("   %-24s %d" % (k, v))

full shape: (885384, 45)
label column: Label | normal value: 'Normal'
class distribution:
   DDoS                     193180
   BruteForce               175925
   UDP Flooding             152775
   MITM                     151586
   ICMP Flooding            47544
   Jamming                  41600
   replay                   25977
   FakeLanding              24050
   Normal                   17949
   De-authentication        14606
   Scanning                 9030
   DoS                      6424
   Reconnassiance           6


In [7]:
# Optional stratified subsample for a tractable first run, then restrict to chosen families
if CONFIG["subsample_n"] and len(df) > CONFIG["subsample_n"]:
    frac = CONFIG["subsample_n"] / len(df)
    df = df.groupby(label_col, group_keys=False).apply(lambda g: g.sample(frac=frac, random_state=42)).reset_index(drop=True)
    print("subsampled to:", df.shape)
if CONFIG["include_families"]:
    keep = [normal_value] + list(CONFIG["include_families"])
    df = df[df[label_col].isin(keep)].reset_index(drop=True)
    families = list(CONFIG["include_families"])
print("families to hold out in turn:", families)

families to hold out in turn: ['DDoS', 'UDP Flooding', 'MITM', 'Jamming', 'BruteForce', 'De-authentication']


In [8]:
# Single held-out-family evaluation for one seed (returns a metrics dict)
def run_once(df, label_col, normal_value, F, seed, cfg):
    S = prepare_splits(df, label_col, normal_value, F, cfg["drop_col_patterns"],
                       cfg["normal_fracs"], cfg["family_fracs"], seed)
    clf = xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                            random_state=seed, **cfg["xgb"])
    clf.fit(S["X_train"], S["y_train"])

    def sc(X):
        p = clf.predict_proba(X)[:, 1]; return p, logit(p)
    p_cal, lg_cal   = sc(S["X_cal"])
    p_seen, lg_seen = sc(S["X_seen"])
    p_shift, lg_shift = sc(S["X_shift"])
    y_seen, y_shift = S["y_seen"], S["y_shift"]

    fitted = fit_calibrators(lg_cal, p_cal, S["y_cal"])
    cs = apply_calibrators(fitted, lg_seen, p_seen)
    ch = apply_calibrators(fitted, lg_shift, p_shift)
    qhat = conformal_qhat(p_cal, S["y_cal"], alpha=cfg["conformal_alpha"])
    pr_s = (p_seen >= 0.5).astype(int)
    pr_h = (p_shift >= 0.5).astype(int)
    nb = cfg["n_ece_bins"]
    return {
        "held_out": F, "seed": seed,
        "shift_heldout_recall": recall_score(y_shift, pr_h, pos_label=1),
        "shift_trivial_base": max(np.mean(y_shift == 0), np.mean(y_shift == 1)),
        "seen_balacc": balanced_accuracy_score(y_seen, pr_s),
        "shift_balacc": balanced_accuracy_score(y_shift, pr_h),
        "seen_macroF1": f1_score(y_seen, pr_s, average="macro"),
        "shift_macroF1": f1_score(y_shift, pr_h, average="macro"),
        "seen_ECE_temp": top_label_ece(cs["temperature"], y_seen, nb),
        "shift_ECE_temp": top_label_ece(ch["temperature"], y_shift, nb),
        "shift_Brier_temp": brier_binary(ch["temperature"], y_shift),
        "seen_coverage": coverage_size(p_seen, y_seen, qhat)[0],
        "shift_coverage": coverage_size(p_shift, y_shift, qhat)[0],
        "seen_AURC": aurc(np.maximum(p_seen, 1 - p_seen), (pr_s == y_seen).astype(float))[0],
        "shift_AURC": aurc(np.maximum(p_shift, 1 - p_shift), (pr_h == y_shift).astype(float))[0],
        "T": fitted["temperature"],
    }

In [ ]:
# Run every chosen family across every seed
rows = []
for seed in CONFIG["seeds"]:
    for F in families:
        rows.append(run_once(df, label_col, normal_value, F, seed, CONFIG))
    print("seed done:", seed)
raw = pd.DataFrame(rows)
print("runs:", raw.shape[0], "= %d seeds x %d families" % (len(CONFIG["seeds"]), len(families)))

seed done: 0
seed done: 1
seed done: 2
seed done: 3
seed done: 4
seed done: 5
seed done: 6
seed done: 7
seed done: 8


In [ ]:
# Aggregate across seeds: mean and std per family
key = ["shift_balacc", "shift_macroF1", "shift_heldout_recall",
       "shift_ECE_temp", "shift_coverage", "shift_AURC", "seen_coverage", "T"]
agg = raw.groupby("held_out")[key].agg(["mean", "std"])
summary = pd.DataFrame(index=agg.index)
for k in key:
    summary[k] = (agg[k]["mean"].round(4).astype(str) + " ± " + agg[k]["std"].round(4).astype(str))
print(summary.to_string())
raw.to_csv(os.path.join(CONFIG["report_dir"], "04_uavnidd_seed_raw.csv"), index=False)
flat = agg.copy(); flat.columns = ["%s_%s" % (a, b) for a, b in flat.columns]
flat.round(6).to_csv(os.path.join(CONFIG["report_dir"], "04_uavnidd_seed_aggregate.csv"))
print("saved 04_uavnidd_seed_raw.csv and 04_uavnidd_seed_aggregate.csv")

In [ ]:
# Figure: per-family spread across seeds on UAV-NIDD (does the UAVIDS pattern replicate?)
fams = list(agg.index)
labels = [str(f)[:14] for f in fams]
panels = [("shift_balacc", "Balanced accuracy (shift)", None),
          ("shift_ECE_temp", "ECE temperature (shift)", None),
          ("shift_coverage", "Conformal coverage (shift)", 1 - CONFIG["conformal_alpha"])]
jit = np.random.default_rng(0)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
for a, (col, title, target) in zip(ax, panels):
    for i, F in enumerate(fams):
        v = raw.loc[raw["held_out"] == F, col].values
        a.scatter(i + jit.uniform(-0.12, 0.12, len(v)), v, s=20, alpha=0.6, color="#6a1b9a")
        a.scatter([i], [v.mean()], marker="_", s=700, color="black", zorder=3)
    a.set_xticks(range(len(fams))); a.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
    a.set_title(title)
    if target is not None:
        a.axhline(target, ls="--", color="gray", lw=1)
fig.suptitle("UAV-NIDD  |  each family held out across %d seeds  |  black bar = mean" % len(CONFIG["seeds"]))
fig.tight_layout()
fig.savefig(os.path.join(CONFIG["fig_dir"], "04_uavnidd_seed_spread.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Commit results (end-of-unit discipline)
!git add reports/ figures/ notebooks/
!git commit -m "04 UAV-NIDD: external-validity replication of multi-family held-out trust evaluation across seeds"
!git push origin main